# Лабораторная работа: классификация ЭКГ

В работе вы пройдёте путь от сырых записей ЭКГ до обученного классификатора диагнозов:

1. загрузите открытый датасет PTB-XL в хранилище ЭКГ `ecg-warehouse`;
2. опишете нужную выборку спецификацией `DatasetSpec`, соберёте и экспортируете датасет;
3. обучите нейросетевой классификатор на PyTorch Lightning и оцените его на тестовой выборке;
4. попробуете улучшить базовую модель.

[PTB-XL 1.0.3](https://physionet.org/content/ptb-xl/1.0.3/) содержит 21 799 двенадцатиканальных ЭКГ длительностью 10 с от 18 869 пациентов. Записи размечены кардиологами кодами стандарта SCP-ECG: диагностическими заключениями (`NORM`, `IMI`, `LVH`, …), заключениями о форме ЭКГ и о ритме (`SR`, `AFIB`, …).

## Варианты

Запишите коды своего варианта в `CLASSES` **в том порядке, в каком они указаны в таблице**: этот порядок задаёт столбцы матрицы меток `y`, выходы сети и порядок классов в метриках.

| Вариант | Коды SCP-ECG (порядок классов) | Состояния |
|:-:|---|---|
| 1 | `NORM`, `AFIB` | норма, фибрилляция предсердий |
| 2 | `NORM`, `CLBBB`, `CRBBB` | норма, полная блокада левой и правой ножки пучка Гиса |
| 3 | `NORM`, `IMI`, `ASMI` | норма, нижний и переднеперегородочный инфаркт миокарда |
| 4 | `NORM`, `LVH`, `LAFB` | норма, гипертрофия левого желудочка, блокада передней ветви левой ножки |
| 5 | `NORM`, `1AVB`, `IRBBB` | норма, АВ-блокада I степени, неполная блокада правой ножки |
| 6 | `SR`, `AFIB`, `STACH`, `SBRAD` | синусовый ритм, фибрилляция предсердий, синусовая тахикардия и брадикардия |
| 7 | `NORM`, `PVC`, `AFIB` | норма, желудочковая экстрасистолия, фибрилляция предсердий |

В варианте 6 все классы — заключения о ритме, поэтому норму в нём представляет код `SR` (синусовый ритм), а не диагностическое заключение `NORM`.

Задача **многометочная** (multi-label): классы не взаимоисключающие, у одной записи может быть несколько кодов варианта сразу (например, `IMI` и `ASMI`) или ни одного. Поэтому у сети по одному независимому выходу на класс, а не softmax по классам.

In [ ]:
!pip install -r requirements.txt

## 1. Данные

Скачайте PTB-XL 1.0.3 и распакуйте.

In [ ]:
!unzip -q -o ptb-xl.zip

In [ ]:
from ecg_warehouse.api import WarehouseAPI
from ecg_warehouse.dataset import DatasetSpec, SplitSpec

### Параметры

- `PTB_XL_PATH` — каталог с распакованным PTB-XL;
- `DATASET_EXPORT_PATH` — куда экспортировать датасет для обучения;
- `CLASSES` — **коды вашего варианта** в порядке из таблицы, например для варианта 2: `CLASSES = ["NORM", "CLBBB", "CRBBB"]`.

In [ ]:
PTB_XL_PATH = "./ptb-xl"
DATASET_EXPORT_PATH = "./dataset"
CLASSES = ["NORM", "AFIB"]

## 2. Хранилище ЭКГ

`ecg-warehouse` хранит метаданные записей (`recordings`) и диагнозы (`diagnoses`) в таблицах Apache Iceberg, а сами сигналы — отдельно, в репозитории Icechunk. `WarehouseAPI` — точка входа для загрузки данных, SQL-запросов и сборки датасетов.

In [ ]:
warehouse_config = {"type": "sql", "uri": "sqlite:///catalog.db", "warehouse": "file://data"}

In [ ]:
wh = WarehouseAPI(config=warehouse_config)

### Загрузка PTB-XL

`ingest_ptb_xl` читает записи с частотой дискретизации 500 Гц (12 отведений × 5000 отсчётов) и их коды SCP-ECG. Загрузка всего датасета занимает заметное время. Если её прервать, повторный запуск в том же окружении пропустит уже загруженные записи.

In [ ]:
wh.ingest_ptb_xl(data_dir=PTB_XL_PATH, rate=500)

`wh.sql` выполняет SQL-запрос (DuckDB) к таблицам `recordings` и `diagnoses` и возвращает `pandas.DataFrame`. Проверим, сколько записей и пациентов загружено.

In [ ]:
wh.sql(
    "SELECT source, sampling_rate_hz, num_leads, count(*) AS recordings, count(DISTINCT patient_id) AS patients FROM recordings GROUP BY ALL"
)

**Задание.** Посчитайте, сколько записей приходится на каждый класс вашего варианта, и оцените баланс классов. Например, для варианта 1:

```python
wh.sql("""
    SELECT code, label_text, count(DISTINCT recording_id) AS recordings
    FROM diagnoses
    WHERE code IN ('NORM', 'AFIB')
    GROUP BY ALL
    ORDER BY recordings DESC
""")
```

## 3. Сборка датасета

`DatasetSpec` декларативно описывает выборку: по спецификации и текущему состоянию таблиц датасет собирается воспроизводимо.

- `name` — имя датасета в реестре хранилища;
- `sources`, `num_leads`, `sampling_rate_hz`, `min_duration_sec`, `max_duration_sec` — фильтры записей: только PTB-XL, 12 отведений, 500 Гц, ровно 10 с;
- `code_systems`, `codes` — оставить только диагнозы SCP-ECG с кодами из `CLASSES`;
- `recording_columns`, `diagnosis_columns` — столбцы метаданных, которые попадут в датасет;
- `split` — разбиение на train и test **по пациентам**: все записи пациента попадают в одну часть, в test уходит около 20 % пациентов, `seed` фиксирует разбиение. Так записи одного человека не окажутся одновременно в обучении и в тесте.

In [ ]:
dataset_spec = DatasetSpec(
    name="test",
    sources=["ptb-xl-1.0.3"],
    num_leads=12,
    sampling_rate_hz=500,
    code_systems=["SCP-ECG"],
    codes=CLASSES,
    min_duration_sec=10,
    max_duration_sec=10,
    recording_columns=[
        "recording_id",
        "source",
        "patient_id",
        "sampling_rate_hz",
        "duration_sec",
        "age_at_recording",
        "sex",
    ],
    diagnosis_columns=[
        "diagnosis_id",
        "recording_id",
        "code",
        "code_system",
        "label_text",
    ],
    split=SplitSpec(test_fraction=0.2, seed=0),
)

`build_dataset` собирает датасет по спецификации и регистрирует его в реестре хранилища под именем и версией. Ячейка возвращает манифест сборки: имя, версию, спецификацию и описание файлов. Сигналы не копируются — датасет ссылается на них в хранилище.

In [ ]:
dataset = wh.build_dataset(spec=dataset_spec)

Экспорт в формат `npz` записывает в `DATASET_EXPORT_PATH`:

- `signals.npz` с массивами `X` — сигналы `float32` формы `(записи, 12, 5000)`; `y` — метки формы `(записи, len(CLASSES))`, где `y[i, j] = 1`, если у записи `i` есть код `CLASSES[j]`; а также `classes`, `recording_id` и `split` (`train` / `test`);
- `meta.csv` — метаданные записей, `diagnoses.csv` — их диагнозы.

In [ ]:
wh.export_dataset(
    dataset_ref=f"{dataset['name']}@{dataset['version']}",
    format="npz",
    output=DATASET_EXPORT_PATH,
)

## 4. Обучение классификатора

Дальше нужны только экспортированные файлы. Обучение построено на PyTorch Lightning и библиотеке `ecg_ml`: `WarehouseDatasetModule` загружает экспорт, `ClassifierLModule` реализует шаги обучения, валидации и теста.

In [ ]:
from functools import partial
import torch
from lightning import Trainer
from ecg_ml.data.warehouse import WarehouseDatasetModule
from ecg_ml.modules.classifier import ClassifierLModule

In [ ]:
LEARNING_RATE = 0.1
BATCH_SIZE = 32
NUM_WORKERS = 2
EPOCHS = 30

### Модель

В ячейке ниже — заготовка `ECGClassifier`.

**Задание.** Замените реализацию `ECGClassifier` в ячейке ниже собственной архитектурой нейросети для классификации ЭКГ. Сохраните имя класса и параметр `output_size`: модель создаётся в ячейке с `ClassifierLModule` как `ECGClassifier(output_size=len(CLASSES))`. Требования к модели:

- вход — батч сигналов формы `(batch, 12, 5000)`: 12 отведений по 5000 отсчётов;
- выход — логиты формы `(batch, len(CLASSES))`.

In [ ]:
class ECGClassifier(torch.nn.Module):
    def __init__(self, output_size=2):
        super().__init__()
        self.flatten = torch.nn.Flatten()
        self.linear = torch.nn.Linear(12 * 5000, output_size)

    def forward(self, x):
        x = self.flatten(x)
        x = self.linear(x)

        return x

### Данные для обучения

In [ ]:
dataset = WarehouseDatasetModule("./dataset")
datamodule = dataset.to_datamodule(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

### Функция потерь и оптимизатор

- `BCEWithLogitsLoss` — бинарная кросс-энтропия, посчитанная независимо для каждого класса; подходит для многометочной задачи;
- `Adam` с шагом обучения `LEARNING_RATE`;
- `ReduceLROnPlateau` уменьшает шаг в 10 раз, если `val/loss` не улучшается 10 эпох подряд.

In [ ]:
optimizer = partial(torch.optim.Adam, lr=LEARNING_RATE)
lr_scheduler = {
    "scheduler": partial(
        torch.optim.lr_scheduler.ReduceLROnPlateau, mode="min", factor=0.1, patience=10
    ),
    "monitor": "val/loss",
    "interval": "epoch",
    "frequency": 1,
}
criterion = torch.nn.BCEWithLogitsLoss()

`ClassifierLModule` объединяет сеть, функцию потерь, оптимизатор и метрики. На валидации и тесте считаются precision, recall и F1 для каждого класса и их macro-среднее.

In [ ]:
classifier = ClassifierLModule(
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    criterion=criterion,
    network=ECGClassifier(output_size=len(CLASSES)),
    labels=CLASSES,
)

### Обучение

`Trainer` проводит валидацию после каждой эпохи и обучает не больше `EPOCHS` эпох в смешанной точности (`precision=16`). Метрики по эпохам сохраняются в `lightning_logs/version_N/metrics.csv`, веса — в `lightning_logs/version_N/checkpoints/`.

In [ ]:
trainer = Trainer(
    precision=16, check_val_every_n_epoch=1, min_epochs=1, max_epochs=EPOCHS
)

In [ ]:
trainer.fit(
    model=classifier,
    datamodule=datamodule,
)

### Оценка на тестовой выборке

`trainer.test` считает метрики на test для весов после последней эпохи и печатает `classification_report` из scikit-learn: precision, recall и F1 по каждому классу. Тестовую выборку используйте только для итоговой оценки, а модели и гиперпараметры сравнивайте по валидации.

In [ ]:
trainer.test(
    model=classifier,
    datamodule=datamodule,
)

## 5. Улучшение модели

Что можно попробовать для улучшения:

- подобрать `LEARNING_RATE`, `BATCH_SIZE`, `EPOCHS`;
- учесть дисбаланс классов: `torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor(dataset.pos_weight))`;